In [18]:
import pandas as pd
import numpy as np
from IPython.display import HTML, display
import plotly.express as px
import plotly.io as pio

# EDITAR: lista de días (ordenada) que quieres incluir en la gráfica
dates = ['20260518', '20260522', '20260603']  # añade más YYYYMMDD según tus carpetas

merged_list = []
for d in dates:
    tr_path = f'../data/experiments/{d}/trays.csv'
    tt_path = f'../data/experiments/{d}/treatment.csv'
    try:
        trays_d = pd.read_csv(tr_path)
        treatment_d = pd.read_csv(tt_path)
    except FileNotFoundError:
        continue

    trays_d['absorption'] = (trays_d['post_weight'] - trays_d['pre_weight']) / trays_d['pre_weight']
    merged_d = trays_d.merge(treatment_d, left_on='treatment_floor', right_on='treatment_id', how='inner')
    merged_d['date'] = d
    merged_list.append(merged_d)

if not merged_list:
    raise RuntimeError("No se cargaron datos para las fechas indicadas.")

merged_all = pd.concat(merged_list, ignore_index=True)

# ordenar por treatment_value y por el orden de dates
date_cat = pd.Categorical(merged_all['date'], categories=dates, ordered=True)
merged_all = merged_all.assign(date_cat=date_cat).sort_values(['treatment_value', 'date_cat', 'tray_id'])

# calculamos un x por (treatment_value, date): todos los puntos de la misma fecha y tratamiento comparten x
n_dates = len(dates)
merged_all['date_idx'] = merged_all['date_cat'].cat.codes  # 0..n_dates-1
spacing = 0.12  # distancia entre fechas (ajusta)
merged_all['x_base'] = merged_all['treatment_value'].astype(float)
merged_all['x_pos'] = merged_all['x_base'] + (merged_all['date_idx'] - (n_dates - 1) / 2) * spacing

# etiqueta mostrada en hover / ticks: '2' para primer día, '2(2)' para segundo día, etc.
day_num = merged_all['date_idx'] + 1
merged_all['x_label'] = np.where(
    day_num == 1,
    merged_all['treatment_value'].astype(str),
    merged_all['treatment_value'].astype(str) + "(" + day_num.astype(str) + ")"
)

# construir figura: colorear por fecha para distinguir días
fig = px.scatter(
    merged_all,
    x='x_pos',
    y='absorption',
    color='date',
    hover_data=['tray_id', 'date', 'treatment_value', 'x_label', 'floor', 'position'],
    title='Treatment (por día alineado) vs Absorption',
    labels={'x_pos': 'Treatment (por fecha)', 'absorption': 'Absorption'}
)

# definir ticks en el eje x con las etiquetas combinadas (una por treatment+date)
tick_map = merged_all.drop_duplicates(subset=['x_pos', 'x_label']).sort_values('x_pos')
fig.update_layout(
    xaxis=dict(
        tickmode='array',
        tickvals=tick_map['x_pos'],
        ticktext=tick_map['x_label'],
        title='Treatment (con fechas separadas)'
    ),
    yaxis=dict(title='Absorption')
)

display(HTML(pio.to_html(fig, include_plotlyjs='cdn')))
